Individual `20ms` observations are too granualar for meaningful state classificaiton.

For this, 
1. taking window size (`W`) = `5 rows` = `100ms` of network activity
2. Stride (`S`) = `1 row` = `20 ms` shift between consecutive windows
3. Resulting overlapping between consecutive windows  = `4 rows (80%)`

`100ms` is long enough to capture short-term trends (singal improving/degrading)

`100ms` is also short enough to enable real-time decisions

In [4]:
import pandas as pd
df = pd.read_csv("DS1.csv")

W = 5 # Window size
S = 1 # Stride
windowed_data = []

KPI_cols = [
    "phy_mcs",
    "mac_dl_cqi",
    "mac_dl_ri",
    "mac_dl_pmi",
    "mac_ul_buffer",
    "mac_n_prb",
    "rsrq",
    "rsrp",
    "rssi",
    "dl_sinr",
    "se",
    "dl_bler",
    "delay"
]

# Sliding Window approach
for i in range(0, len(df)-W+1, S):
    window = df.iloc[i:i+W] # Respective window of each iterations
    sample={} # For storing sample of each window

    for j in range(W):
        for col in KPI_cols:
            sample[f'{col}_{j}'] = window.iloc[j][col]
    windowed_data.append(sample)

final_df = pd.DataFrame(windowed_data)
final_df.to_csv("windowed_output.csv", index=False)


In [ ]:
# df_wo = pd.read_csv("windowed_output.csv")
# df_wo.to_excel("windowed_output.xlsx", index=False)

In future we can add 
1. DL models
2. ML models

according to our need into this

**window-formation with semantic labels**

In [10]:
import pandas as pd
df = pd.read_csv("windowed_output.csv")

# Head 1: link quality
def classify_link_quality(row):
    sinr = row["dl_sinr"]  # SINR: Signal to Inference Noise Ratio
    rsrp = row["rsrp"]  # RSRP: Reference Signal Recieved Power

    if sinr >= 20 and rsrp >= -89:
        return "Excellent"
    elif sinr >= 18 and rsrp >= -91:
        return 'Good'
    elif sinr >= 15 and rsrp >= -95:
        return "Fair"
    else:
        return "Poor"

# Head 2: Congestion


def classify_congestion(row):
    prb = row["mac_n_prb"]  # PRB: physical resource block
    if prb >= 80:
        return "Congested"
    elif prb >= 60:
        return "Busy"
    elif prb >= 30:
        return "Moderate"
    else:
        return "Underutilized"


# Head 3: Latency
def classify_latency(row):
    delay = row["delay"]
    if delay >= 30:
        return "Critical"
    elif delay >= 20:
        return 'Risk'
    elif delay >= 10:
        return "Normal"
    else:
        return "Realtime"


# Head 4: Interference
def classify_interference(row):
    rsrq = row["rsrq"]  # RSRQ: Reference Singla Recieved Quality
    if rsrq >= -8:
        return "Low"
    elif rsrq >= -10:
        return "Moderate"
    else:
        return "Risk"

# Head 5: Reliability


def classify_reliability(row):
    bler = row["dl_bler"]

    if bler >= 5:
        return "Critical"
    elif bler >= 1:
        return "Warning"
    else:
        return "Reliable"


# Head 6: Scheduler
def classify_scheduler(row):
    scheduler = row["mac_dl_cqi"]
    if scheduler >= 9:
        return "Excellent"
    elif scheduler >= 7:
        return "Moderate"
    else:
        return "Poor"
    

for i in range(5):
    level_df = pd.DataFrame({
        "dl_sinr":df[f"dl_sinr_{i}"],
        "rsrp":df[f"rsrp_{i}"],
        "mac_n_prb":df[f"mac_n_prb_{i}"],
        "delay":df[f"delay_{i}"],
        "rsrq":df[f"rsrq_{i}"],
        "dl_bler":df[f"dl_bler_{i}"],
        "mac_dl_cqi":df[f"mac_dl_cqi_{i}"]
    })

    # axis = 1: apply in row-wise format
    df[f"link_quality_{i}"] = level_df.apply(classify_link_quality, axis=1)
    df[f"congestion_{i}"] = level_df.apply(classify_congestion, axis=1)
    df[f"latency_{i}"] = level_df.apply(classify_latency, axis=1)
    df[f"interference_{i}"] = level_df.apply(classify_interference, axis=1)
    df[f"reliability_{i}"] = level_df.apply(classify_reliability, axis=1)
    df[f"scheduler_{i}"] = level_df.apply(classify_scheduler, axis=1)

df.to_csv("windowed_output_semantic.csv", index=False)

In [ ]:
# import pandas as pd
# df = pd.read_csv("windowed_output_semantic.csv")
# df.to_excel("windowed_output_semantic.xlsx", index=False)